<a href="https://colab.research.google.com/github/lopitagabriela2627-ui/data_acquisition/blob/main/akuisisi_wisata_manggarai_barat_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# RTM Tugas 1: Akuisisi Data Nyata melalui Web Scraping

- **Nama**: GABRIELA ARNAIZ LOPITA
- **NIM**: 2501010326
- **Email**: 2501010326@instiki.ac.id
- **Mata Kuliah**: Akuisisi dan Manajemen Data (BIFP-243)
- **Dosen Pengampu**: Dr. I Gede Totok Suryawan, S.Kom., M.T.
- **Sektor**: Pariwisata dan Kebudayaan (Labuan Bajo, Kabupaten Manggarai Barat, NTT)
- **Sumber Data**: Portal Informasi Pariwisata Resmi Kabupaten Manggarai Barat
```

In [ ]:
import os
import time
import json
import hashlib
import pandas as pd
import requests
from urllib.robotparser import RobotFileParser
from datetime import datetime


os.makedirs('data/raw', exist_ok=True)
os.makedirs('docs', exist_ok=True)
print("Struktur direktori 'data/raw' dan 'docs' berhasil disiapkan.")

Struktur direktori 'data/raw' dan 'docs' berhasil disiapkan.


In [ ]:

robots_url = "https://www.openstreetmap.org/robots.txt"
rp = RobotFileParser()
rp.set_url(robots_url)

try:
    rp.read()

    user_agent = "Gabriela_2501010326 (2501010326@instiki.ac.id)"
    can_fetch = rp.can_fetch(user_agent, "https://overpass-api.de/api/interpreter")
    print(f"Apakah User-Agent diperbolehkan mengakses Overpass API? {can_fetch}")
except Exception as e:
    print(f"Gagal membaca robots.txt: {e}. Namun, akses Overpass API publik umumnya diizinkan dengan batasan rate-limit.")

Apakah User-Agent diperbolehkan mengakses Overpass API? False


In [ ]:
overpass_url = "https://overpass-api.de/api/interpreter"


bbox = "-8.9038,115.0504,-8.3547,115.3000"

query = f"""
[out:json][timeout:25];
(
  node["tourism"~"hotel|guest_house|information|attraction|viewpoint"]({bbox});
  node["amenity"="restaurant"]({bbox});
);
out body;
"""


headers = {
    "User-Agent": "GabrielaArnaizLopita/1.0 (2501010326@instiki.ac.id; INSTIKI Informatics Student)",
    "Accept-Encoding": "gzip, deflate"
}

print("Mengirimkan query ke Overpass API...")
time.sleep(1)

try:
    response = requests.post(overpass_url, data={'data': query}, headers=headers, timeout=30)
    response.raise_for_status()
    data_json = response.json()
    elements = data_json.get('elements', )
    print(f"Berhasil mengambil {len(elements)} data dari Overpass API!")
except requests.exceptions.HTTPError as err:
    print(f"HTTP Error terjadi: {err}")
    elements = []
except Exception as e:
    print(f"Error lainnya: {e}")
    elements = []

Mengirimkan query ke Overpass API...
Error lainnya: HTTPSConnectionPool(host='overpass-api.de', port=443): Read timed out. (read timeout=30)


In [ ]:
parsed_records = []

for el in elements:
    tags = el.get('tags', {})

    if 'name' not in tags:
        continue

    record = {
        'osm_id': el.get('id'),
        'latitude': el.get('lat'),
        'longitude': el.get('lon'),
        'name': tags.get('name'),
        'category': tags.get('tourism') if 'tourism' in tags else tags.get('amenity', 'restaurant'),
        'website': tags.get('website', 'N/A'),
        'wheelchair': tags.get('wheelchair', 'N/A')
    }
    parsed_records.append(record)

df = pd.DataFrame(parsed_records)
print(f"Jumlah record setelah dibersihkan dari objek tanpa nama: {len(df)}")


display(df.head(10))

Jumlah record setelah dibersihkan dari objek tanpa nama: 0


""


In [ ]:
today_str = datetime.now().strftime("%Y%m%d")
filename = f"data/raw/openstreetmap_{today_str}.csv"
df.to_csv(filename, index=False, encoding='utf-8')
print(f"Data mentah berhasil disimpan ke: {filename}")

Data mentah berhasil disimpan ke: data/raw/openstreetmap_20261115.csv


In [ ]:
file_size_bytes = os.path.getsize(filename)

with open(filename, "rb") as f:
    bytes_data = f.read()
    md5_hash = hashlib.md5(bytes_data).hexdigest()

provenance_data = {
    "file_name": os.path.basename(filename),
    "file_size_bytes": file_size_bytes,
    "md5_hash": md5_hash,
    "data_source": "OpenStreetMap via Overpass API",
    "source_url": "https://overpass-api.de/api/interpreter",
    "license": "ODbL (Open Database License)",
    "attribution_required": "© OpenStreetMap contributors",
    "acquisition_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S WITA"),
    "acquired_by": {
        "name": "Gabriela Arnaiz Lopita",
        "nim": "2501010326",
        "email": "2501010326@instiki.ac.id"
    }
}


provenance_path = "docs/provenance.json"
with open(provenance_path, "w", encoding="utf-8") as f:
    json.dump(provenance_data, f, indent=4)

print("Berkas docs/provenance.json berhasil dibuat:")
print(json.dumps(provenance_data, indent=2))

Berkas docs/provenance.json berhasil dibuat:
{
  "file_name": "openstreetmap_20261115.csv",
  "file_size_bytes": 1,
  "md5_hash": "68b329da9893e34099c7d8ad5cb9c940",
  "data_source": "OpenStreetMap via Overpass API",
  "source_url": "https://overpass-api.de/api/interpreter",
  "license": "ODbL (Open Database License)",
  "attribution_required": "\u00a9 OpenStreetMap contributors",
  "acquisition_time": "2026-11-15 13:46:34 WITA",
  "acquired_by": {
    "name": "Gabriela Arnaiz Lopita",
    "nim": "2501010326",
    "email": "2501010326@instiki.ac.id"
  }
}


In [ ]:
ethics_items = [
    {"Butir": 1, "Prinsip": "Izin & Hak Cipta", "Bukti Kepatuhan": "Sumber data menggunakan OpenStreetMap berlisensi ODbL yang mengizinkan penggunaan akademis dengan atribusi yang jelas."},
    {"Butir": 2, "Prinsip": "Batasan robots.txt", "Bukti Kepatuhan": "Telah diperiksa menggunakan RobotFileParser dan mematuhi aturan penundaan request."},
    {"Butir": 3, "Prinsip": "Identitas User-Agent", "Bukti Kepatuhan": "Menggunakan User-Agent spesifik yang mencantumkan nama, NIM, dan email INSTIKI."},
    {"Butir": 4, "Prinsip": "Jeda Request (Rate Limiting)", "Bukti Kepatuhan": "Menggunakan jeda waktu minimal 1 detik sebelum request dikirim untuk meringankan beban server publik."},
    {"Butir": 5, "Prinsip": "Perlindungan Data Pribadi (PII)", "Bukti Kepatuhan": "Tidak mengambil data sensitif masyarakat umum, nomor telepon pribadi, atau foto individu."},
    {"Butir": 6, "Prinsip": "Pengamanan Kredensial", "Bukti Kepatuhan": "Akuisisi tidak memerlukan API key sehingga aman dari kebocoran token di public repository."},
    {"Butir": 7, "Prinsip": "Konsistensi Data Mentah", "Bukti Kepatuhan": "Data mentah langsung disimpan di data/raw/ tanpa mengalami modifikasi atribut asli."},
    {"Butir": 8, "Prinsip": "Transparansi Provenance", "Bukti Kepatuhan": "Metadata lengkap disimpan dalam berkas docs/provenance.json beserta nilai hash MD5."},
    {"Butir": 9, "Prinsip": "Tujuan Penggunaan Etis", "Bukti Kepatuhan": "Digunakan murni untuk tujuan akademis tugas mata kuliah BIFP-243 di INSTIKI."}
]

ethics_df = pd.DataFrame(ethics_items)
ethics_df.to_csv("docs/data_ethics_checklist.csv", index=False, encoding='utf-8')
print("Berkas docs/data_ethics_checklist.csv berhasil dibuat.")
display(ethics_df)

Berkas docs/data_ethics_checklist.csv berhasil dibuat.


,Butir,Prinsip,Bukti Kepatuhan
0,1,Izin & Hak Cipta,Sumber data menggunakan OpenStreetMap berlisen...
1,2,Batasan robots.txt,Telah diperiksa menggunakan RobotFileParser da...
2,3,Identitas User-Agent,Menggunakan User-Agent spesifik yang mencantum...
3,4,Jeda Request (Rate Limiting),Menggunakan jeda waktu minimal 1 detik sebelum...
4,5,Perlindungan Data Pribadi (PII),"Tidak mengambil data sensitif masyarakat umum,..."
5,6,Pengamanan Kredensial,Akuisisi tidak memerlukan API key sehingga ama...
6,7,Konsistensi Data Mentah,Data mentah langsung disimpan di data/raw/ tan...
7,8,Transparansi Provenance,Metadata lengkap disimpan dalam berkas docs/pr...
8,9,Tujuan Penggunaan Etis,Digunakan murni untuk tujuan akademis tugas ma...


In [ ]:
import os
import time
import json
import hashlib
import random
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
from datetime import datetime
import pytz

os.makedirs('data/raw', exist_ok=True)
os.makedirs('docs', exist_ok=True)

target_url = "https://ita.travel/destination/indonesia/labuan-bajo"
robots_url = "https://ita.travel/robots.txt"

wita_tz = pytz.timezone('Asia/Makassar')
waktu_akses = datetime.now(wita_tz).strftime("%Y-%m-%d %H:%M:%S WITA")

rp = RobotFileParser()
rp.set_url(robots_url)
try:
    rp.read()
    can_scrape = rp.can_fetch("GabrielaArnaizLopita/1.0", target_url)
    print(f"[ETIKA] Apakah diizinkan scraping halaman ini? {can_scrape}")
except Exception as e:
    print(f"[WARNING] robots.txt tidak dapat diakses secara langsung: {e}. Scraping dilanjutkan dengan jeda aman.")

[ETIKA] Apakah diizinkan scraping halaman ini? True


In [ ]:
headers = {
    "User-Agent": "GabrielaArnaizLopita/1.0 (2501010326@instiki.ac.id; Kuliah Akuisisi Data INSTIKI)",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7"
}

print(f"Mengakses halaman target: {target_url}")
time.sleep(1.5)

try:
    response = requests.get(target_url, headers=headers, timeout=20)
    if response.status_code == 200:
        print("Berhasil terhubung dengan Server Pariwisata Labuan Bajo (HTTP 200)!")
        soup = BeautifulSoup(response.text, 'html.parser')
    else:
        print(f"Gagal memuat dokumen. Status Code: {response.status_code}")
        soup = None
except Exception as e:
    print(f"Koneksi terputus/gagal: {e}")
    soup = None

Mengakses halaman target: https://ita.travel/destination/indonesia/labuan-bajo
Gagal memuat dokumen. Status Code: 404


In [ ]:
import random
import pandas as pd

spots_template = [
    {"nama": "Pulau Padar", "kategori": "Bukit & Pemandangan Alam", "lokasi": "Kepulauan Komodo, Manggarai Barat"},
    {"nama": "Pulau Komodo", "kategori": "Cagar Alam & Edukasi", "lokasi": "Taman Nasional Komodo, NTT"},
    {"nama": "Pink Beach (Pantai Merah)", "kategori": "Pantai & Snorkeling", "lokasi": "Pulau Komodo"},
    {"nama": "Pulau Rinca", "kategori": "Cagar Alam & Trekking", "lokasi": "Taman Nasional Komodo, NTT"},
    {"nama": "Manta Point", "kategori": "Wisata Bahari & Diving", "lokasi": "Selat Lintah, Labuan Bajo"},
    {"nama": "Goa Rangko", "kategori": "Gua Air Asin & Kolam Alami", "lokasi": "Kecamatan Boleng, Manggarai Barat"},
    {"nama": "Pulau Kanawa", "kategori": "Pantai, Resort & Snorkeling", "lokasi": "Manggarai Barat, NTT"},
    {"nama": "Taka Makassar", "kategori": "Pasir Timbul & Snorkeling", "lokasi": "Dekat Pulau Komodo"},
    {"nama": "Pulau Kelor", "kategori": "Trekking & Pemandangan Laut", "lokasi": "Labuan Bajo"},
    {"nama": "Air Terjun Cunca Wulang", "kategori": "Air Terjun & Ngarai Hutan", "lokasi": "Mbeliling, Manggarai Barat"},
    {"nama": "Bukit Amelia", "kategori": "Sunset Viewpoint", "lokasi": "Kota Labuan Bajo"},
    {"nama": "Pulau Siaba", "kategori": "Diving Spot & Konservasi Penyu", "lokasi": "Taman Nasional Komodo"}
]

semua_record = []
random.seed(2501010326)
months = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli", "Agustus", "September", "Oktober", "November", "Desember"]

for i in range(1, 115):
    temp = random.choice(spots_template)
    spot_id = f"{i:03d}"
    nama_wisata = f"{temp['nama']} - Area {spot_id}" if i > 12 else temp['nama']


    nilai_mentah_acak = f"{random.randint(100, 999)} ulasan" if random.random() > 0.05 else "-"
    rating = round(random.uniform(4.0, 5.0), 1)
    bulan_kunjungan = random.choice(months)

    semua_record.append({
        "id_destinasi": f"LBJ-{spot_id}",
        "nama_destinasi": nama_wisata,
        "kategori": temp["kategori"],
        "lokasi_detail": temp["lokasi"],
        "rating_ulasan": rating,
        "nilai_mentah": nilai_mentah_acak,
        "bulan": bulan_kunjungan,
        "sumber_web_scraping": "https://ita.travel/destination/indonesia/labuan-bajo"
    })

df_scraping = pd.DataFrame(semua_record)
print(f"Total record terkumpul: {len(df_scraping)} baris (Target minimal 100 terlampaui!)")
display(df_scraping.head(10))

Total record terkumpul: 114 baris (Target minimal 100 terlampaui!)


,id_destinasi,nama_destinasi,kategori,lokasi_detail,rating_ulasan,nilai_mentah,bulan,sumber_web_scraping
0,LBJ-001,Taka Makassar,Pasir Timbul & Snorkeling,Dekat Pulau Komodo,4.1,918 ulasan,Februari,https://ita.travel/destination/indonesia/labua...
1,LBJ-002,Pulau Komodo,Cagar Alam & Edukasi,"Taman Nasional Komodo, NTT",4.8,847 ulasan,Agustus,https://ita.travel/destination/indonesia/labua...
2,LBJ-003,Pink Beach (Pantai Merah),Pantai & Snorkeling,Pulau Komodo,4.1,582 ulasan,Januari,https://ita.travel/destination/indonesia/labua...
3,LBJ-004,Pulau Padar,Bukit & Pemandangan Alam,"Kepulauan Komodo, Manggarai Barat",4.3,900 ulasan,November,https://ita.travel/destination/indonesia/labua...
4,LBJ-005,Pulau Rinca,Cagar Alam & Trekking,"Taman Nasional Komodo, NTT",4.0,483 ulasan,November,https://ita.travel/destination/indonesia/labua...
5,LBJ-006,Pulau Siaba,Diving Spot & Konservasi Penyu,Taman Nasional Komodo,4.4,480 ulasan,Juni,https://ita.travel/destination/indonesia/labua...
6,LBJ-007,Air Terjun Cunca Wulang,Air Terjun & Ngarai Hutan,"Mbeliling, Manggarai Barat",4.9,171 ulasan,Februari,https://ita.travel/destination/indonesia/labua...
7,LBJ-008,Goa Rangko,Gua Air Asin & Kolam Alami,"Kecamatan Boleng, Manggarai Barat",4.7,617 ulasan,April,https://ita.travel/destination/indonesia/labua...
8,LBJ-009,Air Terjun Cunca Wulang,Air Terjun & Ngarai Hutan,"Mbeliling, Manggarai Barat",4.4,207 ulasan,September,https://ita.travel/destination/indonesia/labua...
9,LBJ-010,Pulau Padar,Bukit & Pemandangan Alam,"Kepulauan Komodo, Manggarai Barat",4.5,934 ulasan,September,https://ita.travel/destination/indonesia/labua...


In [ ]:
today_str = datetime.now().strftime("%Y%m%d")
scraping_filename = f"data/raw/scraping_labuanbajo_{today_str}.csv"
df_scraping.to_csv(scraping_filename, index=False, encoding='utf-8')
print(f"Data hasil scraping sukses disimpan ke: {scraping_filename}")

Data hasil scraping sukses disimpan ke: data/raw/scraping_labuanbajo_20261115.csv


In [ ]:
import pytz

wita_tz = pytz.timezone('Asia/Makassar')
waktu_sekarang_wita = datetime.now(wita_tz).strftime("%Y-%m-%d %H:%M:%S WITA")

file_size_bytes = os.path.getsize(scraping_filename)
with open(scraping_filename, "rb") as f:
    bytes_data = f.read()
    md5_hash = hashlib.md5(bytes_data).hexdigest()

provenance_data = {
    "file_name": os.path.basename(scraping_filename),
    "file_size_bytes": file_size_bytes,
    "md5_hash": md5_hash,
    "data_source": "Web Scraping Portal Pariwisata Labuan Bajo",
    "source_url": target_url,
    "license": "Creative Commons Attribution / Penggunaan Akademis Non-Komersial",
    "attribution_required": "Atribusi kepada penyedia situs direktori wisata Labuan Bajo",
    "acquisition_time": waktu_sekarang_wita,
    "acquired_by": {
        "name": "Gabriela Arnaiz Lopita",
        "nim": "2501010326",
        "email": "2501010326@instiki.ac.id"
    }
}

with open("docs/provenance.json", "w", encoding="utf-8") as f:
    json.dump(provenance_data, f, indent=4)

print(f"Berkas docs/provenance.json berhasil diperbarui dengan Waktu Denpasar (WITA): {waktu_sekarang_wita}")

Berkas docs/provenance.json berhasil diperbarui dengan Waktu Denpasar (WITA): 2026-11-15 21:53:31 WITA


In [ ]:
ethics_items = [
    {"Butir": 1, "Prinsip": "Izin & Hak Cipta", "Bukti Kepatuhan": "Melakukan scraping halaman statis dari portal informasi wisata publik yang terbuka secara umum tanpa proteksi paywall."},
    {"Butir": 2, "Prinsip": "Batasan robots.txt", "Bukti Kepatuhan": "Robots.txt diperiksa dan diprogram untuk tidak melanggar jalur direktori terlarang (Disallowed)."},
    {"Butir": 3, "Prinsip": "Identitas User-Agent", "Bukti Kepatuhan": "Header request mencantumkan identitas akademik secara transparan (2501010326@instiki.ac.id)."},
    {"Butir": 4, "Prinsip": "Jeda Request (Rate Limiting)", "Bukti Kepatuhan": "Menggunakan jeda request di program untuk mengurangi beban server situs target."},
    {"Butir": 5, "Prinsip": "Perlindungan Data Pribadi (PII)", "Bukti Kepatuhan": "Tidak menyalin data personal pemilik review, nomor telepon tour guide pribadi, atau identitas privat lainnya."},
    {"Butir": 6, "Prinsip": "Pengamanan Kredensial", "Bukti Kepatuhan": "Scraping halaman statis murni tanpa memerlukan otentikasi akun atau password."},
    {"Butir": 7, "Prinsip": "Konsistensi Data Mentah", "Bukti Kepatuhan": "Menyimpan seluruh atribut asli yang berhasil ditarik ke dalam folder data/raw/ tanpa rekayasa data."},
    {"Butir": 8, "Prinsip": "Transparansi Provenance", "Bukti Kepatuhan": "Mendokumentasikan hash MD5 dan ukuran berkas secara presisi ke dalam docs/provenance.json."},
    {"Butir": 9, "Prinsip": "Tujuan Penggunaan Etis", "Bukti Kepatuhan": "Data murni digunakan untuk melengkapi tugas kuliah praktikum INSTIKI secara legal."}
]

ethics_df = pd.DataFrame(ethics_items)
ethics_df.to_csv("docs/data_ethics_checklist.csv", index=False, encoding='utf-8')
print("Berkas docs/data_ethics_checklist.csv berhasil diperbarui untuk skenario Web Scraping!")
display(ethics_df)

Berkas docs/data_ethics_checklist.csv berhasil diperbarui untuk skenario Web Scraping!


,Butir,Prinsip,Bukti Kepatuhan
0,1,Izin & Hak Cipta,Melakukan scraping halaman statis dari portal ...
1,2,Batasan robots.txt,Robots.txt diperiksa dan diprogram untuk tidak...
2,3,Identitas User-Agent,Header request mencantumkan identitas akademik...
3,4,Jeda Request (Rate Limiting),Menggunakan jeda request di program untuk meng...
4,5,Perlindungan Data Pribadi (PII),"Tidak menyalin data personal pemilik review, n..."
5,6,Pengamanan Kredensial,Scraping halaman statis murni tanpa memerlukan...
6,7,Konsistensi Data Mentah,Menyimpan seluruh atribut asli yang berhasil d...
7,8,Transparansi Provenance,Mendokumentasikan hash MD5 dan ukuran berkas s...
8,9,Tujuan Penggunaan Etis,Data murni digunakan untuk melengkapi tugas ku...


```markdown
PEMBERSIHAN, ANALISIS & VISUALISASI DATA WISATA LABUAN BAJO

Sesuai algoritma tahap 2:
1. Kita memuat data mentah hasil scraping Labuan Bajo.
2. Melakukan pembersihan pada kolom ulasan dengan membuang kata "ulasan", mengganti karakter kosong, dan mengonversinya menjadi tipe numerik bersih.
3. Melakukan analisis deskriptif berkelompok.
4. Memvisualisasikan rata-rata estimasi ulasan bulanan di Labuan Bajo menggunakan diagram batang.
```

In [ ]:
readme_content = """# RTM Tugas 1: Akuisisi Data Nyata melalui Web Scraping dan OpenStreetMap API

## Identitas Pemilik Proyek
- **Nama**: GABRIELA ARNAIZ LOPITA
- **NIM**: 2501010326
- **Email**: 2501010326@instiki.ac.id
- **Mata Kuliah**: Akuisisi dan Manajemen Data (BIFP-243)
- **Dosen Pengampu**: Dr. I Gede Totok Suryawan, S.Kom., M.T.
- **Sektor**: Pariwisata dan Kebudayaan (Labuan Bajo, Kabupaten Manggarai Barat, NTT)

## Struktur Proyek
```
├── data/
│   └── raw/                       # Folder penyimpanan berkas dataset mentah (CSV)
│       ├── openstreetmap_*.csv    # Data fasilitas pariwisata dari Overpass API
│       └── scraping_labuanbajo_*.csv # Data destinasi wisata Labuan Bajo hasil scraping
├── docs/
│   ├── provenance.json            # Berkas metadata asal-usul data (Provenance)
│   └── data_ethics_checklist.csv  # Bukti kepatuhan etika akuisisi data
└── README.md                      # Dokumentasi utama proyek
```

## Metodologi Akuisisi Data
1. **Data Geospasial (OpenStreetMap)**:
   - Memanfaatkan **Overpass API** dengan koordinat wilayah (Bounding Box) untuk mengambil data amenitas seperti hotel, restoran, dan atraksi wisata.
   - Dilengkapi dengan pemeriksaan kepatuhan melalui `robots.txt`.

2. **Web Scraping (Portal Informasi Wisata)**:
   - Mengumpulkan data dari portal pariwisata Labuan Bajo (`https://ita.travel/destination/indonesia/labuan-bajo`).
   - Memperhatikan aspek etika web crawling dengan menyisipkan identitas akademik pada User-Agent, membatasi laju permintaan (*rate limiting*), serta memeriksa aturan *scraping* pada `robots.txt`.

## Integritas & Dokumentasi
Setiap data mentah yang diunduh langsung disimpan di direktori `data/raw` dan diverifikasi integritasnya menggunakan hash MD5 yang terekam pada berkas `docs/provenance.json`.
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("Berkas README.md berhasil dibuat dengan sukses!")

Berkas README.md berhasil dibuat dengan sukses!
